# Feature Dashboard

This notebook lets you:
1. Run feature extraction and see the output
2. Load an existing `features.csv` and concat new patients onto it
3. Check NaN counts per subject and per feature
4. See exactly which patients are incomplete and what they're missing

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import gzip, shutil
import os

## Settings
Edit these paths to match your setup.

In [2]:
MERGED_DIR     = Path("C:/Users/AKLO0022/EOG_REM/merged_csv_eog")      
FEATURE_CSV    = Path("C:/Users/AKLO0022/EOG_REM/features_csv/features.csv") 
PATIENT_EXCEL  = Path("L:/Auditdata/RBD PD/PD-RBD Glostrup Database_ok/GlostrupRBDData.xlsx")  
LIGHTS_DIR     = Path("L:/Auditdata/RBD PD/PD-RBD Glostrup Database_ok") 
FS             = 250.0
PATTERN        = "*contiguous_eog_merged.csv"

---
## 1. Compress merged CSVs

Run this to compress all merged CSVs in `merged_csv_eog` and delete the uncompressed versions. This is optional but saves a lot of disk space. You can still read the compressed CSVs with `pd.read_csv(..., compression='gzip')`

In [ ]:
# compress_merged.py  (delete after running)

for f in MERGED_DIR.glob(PATTERN):
    gz = f.with_suffix(".csv.gz")
    with open(f, "rb") as fi, gzip.open(gz, "wb", 6) as fo:
        shutil.copyfileobj(fi, fo)
    saved = f.stat().st_size - gz.stat().st_size
    print(f"{f.name}: freed {saved / 1024**2:.1f} MB")
    f.unlink()

---
## 2. Check lights txt
Check for lights.txt files with 0 duration.

In [ ]:
problems = []
for lights_file in sorted(LIGHTS_DIR.rglob("lights.txt")):
    try:
        txt = pd.read_csv(lights_file)
        txt.columns = [c.strip().lower() for c in txt.columns]

        off = txt["lights_off"].iloc[0]
        on  = txt["lights_on"].iloc[0]

        # Check for NaN or zero values 
        off_zero = pd.isna(off) or float(off) == 0.0
        on_zero = pd.isna(on) or float(on) == 0.0

        if off_zero and on_zero:
            problems.append({"session": lights_file.parent.name, "lights_off": off, "lights_on": on, "issue": "both zero"})
        elif off_zero:
            problems.append({"session": lights_file.parent.name, "lights_off": off, "lights_on": on, "issue": "lights_off is zero"})
        elif on_zero:
            problems.append({"session": lights_file.parent.name, "lights_off": off, "lights_on": on, "issue": "lights_on is zero"})
    except Exception as e:
        problems.append({"session": lights_file.parent.name, "lights_off": None, "lights_on": None, "issue": str(e)})

df_lights = pd.DataFrame(problems)
print(f"Problematic lights files: {len(df_lights)} / {len(list(LIGHTS_DIR.rglob('lights.txt')))}\n")
print(df_lights.to_string(index=False))

---
## 3. NaN inspection
Load `features.csv` and check for missing values.

In [3]:
df = pd.read_csv(FEATURE_CSV)
print(f"Loaded: {df.shape[0]} subjects, {df.shape[1]} columns")

Loaded: 219 subjects, 147 columns


### 3a. NaN count per feature
Which features have the most missing values?

In [4]:
nan_per_feature = df.isna().sum()
nan_per_feature = nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)

if nan_per_feature.empty:
    print("No NaN values in any feature!")
else:
    print(f"{len(nan_per_feature)} features have NaN values:\n")
    for feat, count in nan_per_feature.items():
        pct = count / len(df) * 100
        print(f"  {feat:<45s}  {count:3d} / {len(df)}  ({pct:.1f}%)")

50 features have NaN values:

  rem_event_mean_roc_amp_uv                       15 / 219  (6.8%)
  rem_event_mean_duration_s                       15 / 219  (6.8%)
  rem_event_median_duration_s                     15 / 219  (6.8%)
  rem_event_mean_loc_amp_uv                       15 / 219  (6.8%)
  rem_event_mean_roc_rise_slope                   15 / 219  (6.8%)
  rem_event_mean_loc_rise_slope                   15 / 219  (6.8%)
  em_amplitude_variance                           11 / 219  (5.0%)
  eeg_theta_delta_ratio_phasic                     9 / 219  (4.1%)
  rem_em_mean_amp_uv                               8 / 219  (3.7%)
  rem_em_mean_duration_s                           8 / 219  (3.7%)
  eeg__n1__theta_ratio                             8 / 219  (3.7%)
  eeg__n1__total                                   8 / 219  (3.7%)
  eeg__n1__beta                                    8 / 219  (3.7%)
  eeg__n1__alpha                                   8 / 219  (3.7%)
  eeg__n1__theta                

### 3b. NaN count per subject
Which subjects have missing values, and how many?

In [ ]:
id_cols = ["subject_id", "DCSM_ID"]
feature_cols = [c for c in df.columns if c not in id_cols and pd.api.types.is_numeric_dtype(df[c])]

nan_per_subject = df[feature_cols].isna().sum(axis=1)
subjects_with_nan = df[nan_per_subject > 0][["subject_id"]].copy()
subjects_with_nan["nan_count"] = nan_per_subject[nan_per_subject > 0].values

if subjects_with_nan.empty:
    print("All subjects have complete features!")
else:
    print(f"{len(subjects_with_nan)} subject(s) have NaN values:\n")
    for _, row in subjects_with_nan.sort_values("nan_count", ascending=False).iterrows():
        print(f"  {row['subject_id']:<50s}  {row['nan_count']:.0f} missing features")

### 3c. Detailed NaN report
For each subject with NaN values, show exactly which features are missing.

In [ ]:
rows_with_nan = df[df[feature_cols].isna().any(axis=1)]

if rows_with_nan.empty:
    print("No NaN values found.")
else:
    for _, row in rows_with_nan.iterrows():
        sid = row["subject_id"]
        missing = [c for c in feature_cols if pd.isna(row[c])]
        print(f"\n{sid}  ({len(missing)} missing):")
        for col in missing:
            print(f"    - {col}")

### 3d. Group label check
Check which subjects have group labels (Control, iRBD, etc.) and which don't.

In [7]:
group_cols = [c for c in ["Control", "PD(-RBD)", "PD(+RBD)", "iRBD", "PLM"] if c in df.columns]

if not group_cols:
    print("No group columns found. Run extract with --patient-excel to add them.")
else:
    has_label = df[group_cols].sum(axis=1) > 0
    n_labelled = has_label.sum()
    n_unlabelled = (~has_label).sum()
    
    print(f"With group label:     {n_labelled}")
    print(f"Without group label:  {n_unlabelled}")
    
    if n_unlabelled > 0:
        unlabelled = df.loc[~has_label, "subject_id"].tolist()
        print(f"\nUnlabelled subjects:")
        for sid in unlabelled:
            print(f"  - {sid}")
    
    # Group distribution
    print(f"\nGroup distribution:")
    for col in group_cols:
        n = (df[col] == 1).sum()
        print(f"  {col:<12s}  n = {n}")

With group label:     219
Without group label:  0

Group distribution:
  Control       n = 52
  PD(-RBD)      n = 35
  PD(+RBD)      n = 52
  iRBD          n = 39
  PLM           n = 41


---
## 4. Quick summary

In [8]:
print(f"Total subjects:       {len(df)}")
print(f"Total features:       {len(feature_cols)}")
print(f"Subjects with NaN:    {len(rows_with_nan)}")
print(f"Features with NaN:    {len(nan_per_feature)}")
print(f"Total NaN values:     {df[feature_cols].isna().sum().sum()}")

Total subjects:       219
Total features:       145
Subjects with NaN:    39
Features with NaN:    50
Total NaN values:     295


---
## 5. ID cleanup

#### Function:

In [25]:
def clean_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    After collect_features + merge_patient_info, the DataFrame may contain:
      - subject_id  
      - DCSM_ID    
      - Duplicate group columns from patient module AND Excel join
        (Control, PD(-RBD), PD(+RBD), iRBD, PLM — possibly with _x/_y suffixes)

    This function consolidates duplicates and drops redundant columns.
    """
    df = df.copy()

    # 1) Drop DCSM_ID if it matches subject_id
    if "DCSM_ID" in df.columns:
            df.drop(columns=["DCSM_ID"], inplace=True)
            print("Dropped 'DCSM_ID' (identical to subject_id)")

    # 2) Consolidate _x / _y suffix duplicates from pd.merge
    suffixed = {}
    for col in df.columns:
        for suffix in ("_x", "_y"):
            if col.endswith(suffix):
                base = col[: -len(suffix)]
                suffixed.setdefault(base, []).append(col)

    for base, cols in suffixed.items():
        # Keep _x, fill NaNs from _y, drop both suffixed versions
        primary = f"{base}_x"
        secondary = f"{base}_y"
        if primary in df.columns and secondary in df.columns:
            df[base] = df[primary].fillna(df[secondary])
            df.drop(columns=[primary, secondary], inplace=True)
            print(f"Merged '{primary}' + '{secondary}' → '{base}'")

    # 3) Drop any remaining _dup columns from the outer join
    dup_cols = [c for c in df.columns if c.endswith("_dup")]
    if dup_cols:
        df.drop(columns=dup_cols, inplace=True)
        print(f"Dropped _dup columns: {dup_cols}")

    print(f"\nFinal shape: {df.shape[0]} subjects × {df.shape[1]} columns")
    return df

#### Run function:

In [26]:
df = pd.read_csv(FEATURE_CSV)                             # Load the original features CSV 
print(f"Before: {df.shape}")                              # Print shape before cleaning
df = clean_feature_columns(df)                            # Clean up redundant ID / label columns after merge
print(f"After: {df.shape}")                               # Print shape after cleaning
df.to_csv("features_csv/features_clean.csv", index=False) # Save the cleaned DataFrame to a new CSV file
print("Cleaned features saved to 'features_csv/features_clean.csv'")

Before: (221, 147)
Dropped 'DCSM_ID' (identical to subject_id)

Final shape: 221 subjects × 146 columns
After: (221, 146)
Cleaned features saved to 'features_csv/features_clean.csv'


---
## 6. Remove invalid lights subjects from feature CSV

Uses the problematic sessions found in Section 2 (`df_lights`).
You can also add extra IDs manually in `EXTRA_EXCLUDE`.

Set `DRY_RUN = False` to actually write the file.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
EXTRA_EXCLUDE = []    # e.g. ['DCSM_999_a'] — add any extra IDs here
OUTPUT_CSV    = Path("features_clean.csv")   # overwrites in place; change path to keep original
DRY_RUN       = False          # set False to actually write
# ────────────────────────────────────────────────────────────────────

# Collect IDs from Section 2 lights check + any manual extras
lights_ids = df_lights["session"].tolist() if "df_lights" in dir() and not df_lights.empty else []
exclude_ids = list(set(lights_ids + EXTRA_EXCLUDE))

if not exclude_ids:
    print("Nothing to exclude — df_lights is empty and EXTRA_EXCLUDE is empty.")
else:
    # Load fresh from disk so this cell is independent of Section 3/5
    df_src = pd.read_csv(FEATURE_CSV)
    id_col = "subject_id" if "subject_id" in df_src.columns else "DCSM_ID"

    # Show what will be removed
    to_remove = df_src[df_src[id_col].isin(exclude_ids)]
    not_found = [i for i in exclude_ids if i not in df_src[id_col].values]

    print(f"Subjects to remove ({len(to_remove)}):")
    for _, row in to_remove.iterrows():
        print(f"  {row[id_col]:<20s}  reason: lights issue")

    if not_found:
        print(f"\nNot found in CSV (already removed or wrong ID?): {not_found}")

    df_clean = df_src[~df_src[id_col].isin(exclude_ids)].reset_index(drop=True)

    print(f"\nBefore : {len(df_src)} subjects")
    print(f"Removed: {len(to_remove)}")
    print(f"After  : {len(df_clean)} subjects")

    if DRY_RUN:
        print(f"\n[DRY RUN] No file written. Set DRY_RUN=False to save to:\n  {OUTPUT_CSV}")
    else:
        df_clean.to_csv(OUTPUT_CSV, index=False)
        print(f"\nSaved → {OUTPUT_CSV}")
        df = df_clean  # update in-memory df for any cells run after this